In [1]:
%load_ext autoreload
%autoreload 2

# Connect to websocket done

In [2]:
# In dev.ipynb
import asyncio
import websockets
import json

async def test_and_decode():
    async with websockets.connect("ws://localhost:8000/ws") as websocket:
        print("✅ Connected!")
        
        message = "Hello from Jupyter!"
        await websocket.send(message)
        print(f"📤 Sent: {message}")
        
        response = await websocket.recv()
        print(f"📥 Raw response: {response}")
        
        # Parse and decode the JSON
        try:
            data = json.loads(response)
            print("\n🔍 Parsed Response:")
            print(f"Type: {data['type']}")
            print(f"Stage: {data['stage_name']}")
            
            guide = data['guide']
            print(f"\nAction: {guide['action']}")
            print(f"Explanation: {guide['explanation']}")
            print(f"Signals: {guide['signals']}")
            print(f"Lines to say: {guide['lines_to_say']}")
            
        except Exception as e:
            print(f"Failed to parse: {e}")

await test_and_decode()

✅ Connected!
📤 Sent: Hello from Jupyter!
📥 Raw response: {"type": "guide", "stage_name": "greeting", "guide": {"action": "\u0e40\u0e23\u0e34\u0e48\u0e21\u0e01\u0e32\u0e23\u0e2a\u0e19\u0e17\u0e19\u0e32\u0e14\u0e49\u0e27\u0e22\u0e01\u0e32\u0e23\u0e17\u0e31\u0e01\u0e17\u0e32\u0e22\u0e2d\u0e22\u0e48\u0e32\u0e07\u0e21\u0e37\u0e2d\u0e2d\u0e32\u0e0a\u0e35\u0e1e\u0e41\u0e25\u0e30\u0e41\u0e19\u0e30\u0e19\u0e33\u0e15\u0e31\u0e27", "explanation": "\u0e40\u0e23\u0e34\u0e48\u0e21\u0e01\u0e32\u0e23\u0e42\u0e17\u0e23\u0e42\u0e14\u0e22\u0e01\u0e32\u0e23\u0e2a\u0e23\u0e49\u0e32\u0e07\u0e04\u0e27\u0e32\u0e21\u0e19\u0e48\u0e32\u0e40\u0e0a\u0e37\u0e48\u0e2d\u0e16\u0e37\u0e2d\u0e41\u0e25\u0e30\u0e2a\u0e23\u0e49\u0e32\u0e07\u0e04\u0e27\u0e32\u0e21\u0e2a\u0e31\u0e21\u0e1e\u0e31\u0e19\u0e18\u0e4c\u0e01\u0e31\u0e1a\u0e25\u0e39\u0e01\u0e04\u0e49\u0e32", "signals": ["\u0e40\u0e1e\u0e34\u0e48\u0e21\u0e40\u0e23\u0e34\u0e48\u0e21\u0e15\u0e49\u0e19\u0e01\u0e32\u0e23\u0e42\u0e17\u0e23", "\u0e25\u0e39\u0e01\u0e04\u0e4

# Each agent run fine

In [ ]:
from package.prompt_hub import PromptHub
from package.llms.bedrock import BedrockNova
from package.llms.extractor import parse_blockcode, Extractor
from package.datamodel.agent_datamodel import CustomerInfo, AgentCheckList, CustomerInterest, Guide

In [ ]:
def run_extractor(model_id, agent_name, system_prompt, DataModel, format, content):
    llm = BedrockNova(model_id=model_id)
    extractor = Extractor(
        agent_name=agent_name,
        llm=llm,
        system_prompt=system_prompt,
        DataModel=DataModel,
        format=format
    )
    response = extractor.run([dict(role="user", content=content)])
    return response

In [15]:
model_id="us.amazon.nova-micro-v1:0"

In [17]:
response = run_extractor(
    model_id=model_id, 
    agent_name="customer_information_extractor",
    system_prompt=PromptHub().extract_agent_checklist,
    DataModel=AgentCheckList,
    format="toon",
    content="สวัสดีครับผมแบงค์จากบริษัทประกันชีวิต ขออนุญาตเรียนสายพี่สมชายเพื่อแนะนำผลิตภัณฑ์ประกันชีวิตของเราครับ"
)
response

AgentCheckList(agent_introduced=True, company_mentioned=True, permission_asked=True)

In [56]:
response = run_extractor(
    model_id=model_id, 
    agent_name="customer_information_extractor",
    system_prompt=PromptHub().extract_customer_information,
    DataModel=CustomerInfo,
    format="toon",
    content="พี่สมชายตอนนี้อายุเท่าไหร่แล้วครับ? ตอนนี้ผมอายุ 45 ปีครับ"
)
response

CustomerInfo(age=45, income_per_month=None, marital_status=None, number_of_children=None)

In [60]:
response.model_dump()

{'age': 45,
 'income_per_month': None,
 'marital_status': None,
 'number_of_children': None}

In [20]:
response = run_extractor(
    model_id=model_id, 
    agent_name="customer_information_extractor",
    system_prompt=PromptHub().extract_customer_interest,
    DataModel=CustomerInterest,
    format="toon",
    content="ตอนนี้พี่สมชายสนใจผลิตภัณฑ์ประกันชีวิตของเรารึเปล่าครับ? สนใจครับผมอยากรู้รายละเอียดของผลิตภัณฑ์ประกันชีวิตของคุณครับ"
)
response

CustomerInterest(life_insurance=True, health_insurance=None, critical_illness=None, accident_insurance=None, retirement_planning=None, tax_benefits=None)

In [22]:
response = run_extractor(
    model_id=model_id, 
    agent_name="customer_information_extractor",
    system_prompt=PromptHub().objection_handling_agent,
    DataModel=Guide,
    format="json",
    content="ตอนนี้ผมยังไม่กล้าตัดสินใจครับ อยากจะขอเวลาคิดดูก่อนครับ"
)
response

Guide(action='ยอมรับความกังวลของลูกค้าและให้เวลาคิด', explanation='การยอมรับความกังวลและให้เวลาคิดแก่ลูกค้าจะช่วยลดความตึงเครียดและสร้างความมั่นใจว่าเราให้ความสำคัญกับความต้องการของลูกค้า', signals=['ลูกค้าแสดงความไม่แน่ใจ', 'ลูกค้าขอเวลาคิดดูก่อน'], lines_to_say=['ไม่เป็นไรเลย เราให้เวลาคุณเพื่อนำความคิดไปพิจารณากับแฟนหรือครอบครัวของคุณ เรารออยู่ที่นี่สำหรับคุณได้เสมอ', 'คุณมีเวลาที่จะคิดดูให้เต็มใจ เราจะรอฟังความเห็นของคุณตอนที่คุณพร้อม'])

In [23]:
response = run_extractor(
    model_id=model_id, 
    agent_name="customer_information_extractor",
    system_prompt=PromptHub().discovery_agent,
    DataModel=Guide,
    format="json",
    content="ตอนนี้พี่สมชายมีแบบประกันอะไรอยู่บ้างครับ แล้วไม่ทราบว่ามีความสนใจในแบบประกันแบบไหนอยู่หรือเปล่าครับ"
)
response

Guide(action='สอบถามข้อมูลปัจจุบันและความสนใจ', explanation='การสอบถามเกี่ยวกับประกันที่มีอยู่และความสนใจในประเภทประกันต่าง ๆ จะช่วยเข้าไปเข้าใจความต้องการและปัญหาที่ต้องแก้ไขของลูกค้าได้ดีกว่า', signals=['ลูกค้าพูดถึงประกันที่มีอยู่', 'ลูกค้าสอบถามความสนใจในประกัน'], lines_to_say=['ขอบคุณสำหรับข้อมูลที่ให้ เราสามารถช่วยค้นหาประกันที่เหมาะสมสำหรับคุณได้', 'คุณมีความสนใจในประเภทประกันใดอยู่หรือไม่ครับ เพื่อที่เราจะเข้าใจความต้องการของคุณได้ดีกว่า'])

In [24]:
response = run_extractor(
    model_id=model_id, 
    agent_name="customer_information_extractor",
    system_prompt=PromptHub().pitch_agent,
    DataModel=Guide,
    format="json",
    content="ตอนนี้ผมว่าพี่สมชายอาจจะสนใจแบบประกันสุขภาพของเรานะครับ เพราะว่าจากที่ผมได้สอบถามมา พี่สมชายมีความกังวลเกี่ยวกับสุขภาพอยู่บ้าง แล้วแบบประกันสุขภาพของเราก็ครอบคลุมค่าใช้จ่ายในการรักษาพยาบาลได้ค่อนข้างดีเลยครับ ได้ครับงั้นผมต้องทำอย่างไรต่อครับ"
)
response

Guide(action='เน้นคุณค่าของประกันสุขภาพด้วยตัวอย่างและสร้างความมั่นใจให้ลูกค้า', explanation='การเน้นคุณค่าด้วยตัวอย่างจะช่วยให้ลูกค้าเข้าใจได้ง่ายว่าประกันสุขภาพจริงๆ ช่วยได้อย่างไร และสร้างความมั่นใจให้ลูกค้าที่กังวลเรื่องสุขภาพ', signals=['ลูกค้ามีความกังวลเกี่ยวกับสุขภาพ', 'ลูกค้าสนใจแบบประกันสุขภาพ'], lines_to_say=['พี่สมชาย, เรามีแผนประกันสุขภาพที่ครอบคลุมการรักษาพยาบาลด้านสุขภาพอย่างครบถ้วน เช่นการรักษาโรคเรื้อรังและการผ่าตัดที่คุณจะต้องเผชิญอยู่', 'สามารถใช้ประกันได้ทันทีและค่าใช้จ่ายที่เราครอบคลุมก็มีการควบคุมให้คุณไม่ต้องกังวลเรื่องเงินมากนัก'])

In [25]:
response = run_extractor(
    model_id=model_id, 
    agent_name="customer_information_extractor",
    system_prompt=PromptHub().closing_agent,
    DataModel=Guide,
    format="json",
    content="ตอนนี้ผมว่าพี่สมชายอาจจะสนใจแบบประกันสุขภาพของเรานะครับ เพราะว่าจากที่ผมได้สอบถามมา พี่สมชายมีความกังวลเกี่ยวกับสุขภาพอยู่บ้าง แล้วแบบประกันสุขภาพของเราก็ครอบคลุมค่าใช้จ่ายในการรักษาพยาบาลได้ค่อนข้างดีเลยครับ ได้ครับงั้นผมต้องทำอย่างไรต่อครับ"
)
response

Guide(action='แนะนำขั้นตอนการสมัคร', explanation='ลูกค้าแสดงความสนใจและสอบถามถึงขั้นตอนการสมัคร ซึ่งเป็นสัญญาณว่าพร้อมดำเนินการต่อ', signals=['ลูกค้าพร้อมดำเนินการสมัคร', 'ลูกค้าสอบถามขั้นตอนการสมัคร'], lines_to_say=['ด้วยความยินดีที่ลูกค้าสนใจแบบประกันสุขภาพของเรา ขั้นตอนต่อไปคือการกรอกแบบฟอร์มข้อมูลส่วนบุคคลของลูกค้า แล้วเราจะทำการตรวจสอบและอนุมัติให้ลูกค้าภายในไม่กี่วัน', 'ลองเริ่มต้นด้วยการกรอกข้อมูลต่าง ๆ ที่เราต้องการเพื่อดำเนินการสมัคร จากนั้นเราก็จะเตรียมส่งใบเสนอราคาเพื่อลูกค้าพิจารณา'])

In [26]:
# In dev.ipynb
import asyncio
import websockets

async def send_and_wait(message):
    async with websockets.connect("ws://localhost:8000/ws") as websocket:
        # Send message
        await websocket.send(message)
        
        # Wait for response
        response = await websocket.recv()
        
        return response

# Usage
response = await send_and_wait("Hello!")
print(response)


{"type": "guide", "stage_name": "greeting", "guide": {"action": "\u0e40\u0e23\u0e34\u0e48\u0e21\u0e01\u0e32\u0e23\u0e2a\u0e19\u0e17\u0e19\u0e32\u0e14\u0e49\u0e27\u0e22\u0e01\u0e32\u0e23\u0e17\u0e31\u0e01\u0e17\u0e32\u0e22\u0e2d\u0e22\u0e48\u0e32\u0e07\u0e21\u0e37\u0e2d\u0e2d\u0e32\u0e0a\u0e35\u0e1e\u0e41\u0e25\u0e30\u0e41\u0e19\u0e30\u0e19\u0e33\u0e15\u0e31\u0e27", "explanation": "\u0e40\u0e23\u0e34\u0e48\u0e21\u0e01\u0e32\u0e23\u0e42\u0e17\u0e23\u0e42\u0e14\u0e22\u0e01\u0e32\u0e23\u0e2a\u0e23\u0e49\u0e32\u0e07\u0e04\u0e27\u0e32\u0e21\u0e19\u0e48\u0e32\u0e40\u0e0a\u0e37\u0e48\u0e2d\u0e16\u0e37\u0e2d\u0e41\u0e25\u0e30\u0e2a\u0e23\u0e49\u0e32\u0e07\u0e04\u0e27\u0e32\u0e21\u0e2a\u0e31\u0e21\u0e1e\u0e31\u0e19\u0e18\u0e4c\u0e01\u0e31\u0e1a\u0e25\u0e39\u0e01\u0e04\u0e49\u0e32", "signals": ["\u0e40\u0e1e\u0e34\u0e48\u0e21\u0e40\u0e23\u0e34\u0e48\u0e21\u0e15\u0e49\u0e19\u0e01\u0e32\u0e23\u0e42\u0e17\u0e23", "\u0e25\u0e39\u0e01\u0e04\u0e49\u0e32\u0e23\u0e31\u0e1a\u0e2a\u0e32\u0e22\u0e41\u0e25\u

In [35]:
# In dev.ipynb
import asyncio
import websockets

class WSClient:
    def __init__(self, uri="ws://localhost:8000/ws"):
        self.uri = uri
        self.websocket = None
    
    async def connect(self):
        self.websocket = await websockets.connect(self.uri)
    
    async def send_and_wait(self, message):
        await self.websocket.send(message)
        response = await self.websocket.recv()
        return response
    
    async def close(self):
        await self.websocket.close()

# # Usage
# client = WSClient()
# await client.connect()

# response1 = await client.send_and_wait("Message 1")
# response2 = await client.send_and_wait("Message 2")

# print(response1)
# print(response2)

# await client.close()


In [62]:
# In dev.ipynb
import asyncio
import websockets
import json

class WSClient:
    def __init__(self, uri="ws://localhost:8000/ws"):
        self.uri = uri
        self.websocket = None
    
    async def connect(self):
        self.websocket = await websockets.connect(self.uri)
        # Receive initial greeting guide
        initial_response = await self.websocket.recv()
        print("📥 Initial greeting received:")
        print(json.dumps(json.loads(initial_response), indent=2))
        return initial_response
    
    async def send_json_and_wait(self, message_dict):
        json_message = json.dumps(message_dict)
        await self.websocket.send(json_message)
        response = await self.websocket.recv()
        return json.loads(response)
    
    async def send_and_wait(self, message):
        await self.websocket.send(message)
        response = await self.websocket.recv()
        return response
    
    async def close(self):
        await self.websocket.close()

In [63]:
client = WSClient()
await client.connect()

📥 Initial greeting received:
{
  "type": "guide",
  "stage_name": "greeting",
  "guide": {
    "action": "\u0e40\u0e23\u0e34\u0e48\u0e21\u0e01\u0e32\u0e23\u0e2a\u0e19\u0e17\u0e19\u0e32\u0e14\u0e49\u0e27\u0e22\u0e01\u0e32\u0e23\u0e17\u0e31\u0e01\u0e17\u0e32\u0e22\u0e2d\u0e22\u0e48\u0e32\u0e07\u0e21\u0e37\u0e2d\u0e2d\u0e32\u0e0a\u0e35\u0e1e\u0e41\u0e25\u0e30\u0e41\u0e19\u0e30\u0e19\u0e33\u0e15\u0e31\u0e27",
    "explanation": "\u0e40\u0e23\u0e34\u0e48\u0e21\u0e01\u0e32\u0e23\u0e42\u0e17\u0e23\u0e42\u0e14\u0e22\u0e01\u0e32\u0e23\u0e2a\u0e23\u0e49\u0e32\u0e07\u0e04\u0e27\u0e32\u0e21\u0e19\u0e48\u0e32\u0e40\u0e0a\u0e37\u0e48\u0e2d\u0e16\u0e37\u0e2d\u0e41\u0e25\u0e30\u0e2a\u0e23\u0e49\u0e32\u0e07\u0e04\u0e27\u0e32\u0e21\u0e2a\u0e31\u0e21\u0e1e\u0e31\u0e19\u0e18\u0e4c\u0e01\u0e31\u0e1a\u0e25\u0e39\u0e01\u0e04\u0e49\u0e32",
    "signals": [
      "\u0e40\u0e1e\u0e34\u0e48\u0e21\u0e40\u0e23\u0e34\u0e48\u0e21\u0e15\u0e49\u0e19\u0e01\u0e32\u0e23\u0e42\u0e17\u0e23",
      "\u0e25\u0e39\u0e01\u0e04

'{"type": "guide", "stage_name": "greeting", "guide": {"action": "\\u0e40\\u0e23\\u0e34\\u0e48\\u0e21\\u0e01\\u0e32\\u0e23\\u0e2a\\u0e19\\u0e17\\u0e19\\u0e32\\u0e14\\u0e49\\u0e27\\u0e22\\u0e01\\u0e32\\u0e23\\u0e17\\u0e31\\u0e01\\u0e17\\u0e32\\u0e22\\u0e2d\\u0e22\\u0e48\\u0e32\\u0e07\\u0e21\\u0e37\\u0e2d\\u0e2d\\u0e32\\u0e0a\\u0e35\\u0e1e\\u0e41\\u0e25\\u0e30\\u0e41\\u0e19\\u0e30\\u0e19\\u0e33\\u0e15\\u0e31\\u0e27", "explanation": "\\u0e40\\u0e23\\u0e34\\u0e48\\u0e21\\u0e01\\u0e32\\u0e23\\u0e42\\u0e17\\u0e23\\u0e42\\u0e14\\u0e22\\u0e01\\u0e32\\u0e23\\u0e2a\\u0e23\\u0e49\\u0e32\\u0e07\\u0e04\\u0e27\\u0e32\\u0e21\\u0e19\\u0e48\\u0e32\\u0e40\\u0e0a\\u0e37\\u0e48\\u0e2d\\u0e16\\u0e37\\u0e2d\\u0e41\\u0e25\\u0e30\\u0e2a\\u0e23\\u0e49\\u0e32\\u0e07\\u0e04\\u0e27\\u0e32\\u0e21\\u0e2a\\u0e31\\u0e21\\u0e1e\\u0e31\\u0e19\\u0e18\\u0e4c\\u0e01\\u0e31\\u0e1a\\u0e25\\u0e39\\u0e01\\u0e04\\u0e49\\u0e32", "signals": ["\\u0e40\\u0e1e\\u0e34\\u0e48\\u0e21\\u0e40\\u0e23\\u0e34\\u0e48\\u0e21\\u0e15\\u0e49\\u

In [64]:
# Test individual commands quickly
async def quick_test(client, command_type, data):
    # client = WSClient()
    # await client.connect()
    
    command = {"type": command_type, "data": data}
    response = await client.send_json_and_wait(command)
    
    print(f"Command: {command_type}")
    print(f"Response: {response}")
    
    # await client.close()

# Quick tests
# await quick_test(client, "manual_information_update", {"age": 30})
await quick_test(client, "guide", {"stage_name": "discovery", "content": "ผมชื่อสมชายอายุ 24 ปีครับ"})
# await quick_test(client, "manual_resolve_objection", {"resolved": True})


Command: guide
Response: {'type': 'guide', 'stage_name': 'discovery', 'message': 'Processing conversation for discovery stage', 'status': 'processing'}


In [65]:
client.close()

<coroutine object WSClient.close at 0x00000207318B6EC0>

In [66]:
# In dev.ipynb
async def collect_all_responses(client, command):
    await client.websocket.send(json.dumps(command))
    
    responses = []
    while True:
        try:
            response = await asyncio.wait_for(client.websocket.recv(), timeout=5.0)
            data = json.loads(response)
            responses.append(data)
            print(f"📥 Got: {data['type']}")
        except asyncio.TimeoutError:
            break
    
    return responses

# Usage
client = WSClient()
await client.connect()

all_responses = await collect_all_responses(client, {
    "type": "guide",
    "data": {"stage_name": "discovery", "content": "ผมชื่อสมชายอายุ 24 ปีครับ"}
})

print(f"Total responses: {len(all_responses)}")
for i, resp in enumerate(all_responses):
    print(f"{i+1}. {resp['type']}: {resp}")

await client.close()


📥 Initial greeting received:
{
  "type": "guide",
  "stage_name": "greeting",
  "guide": {
    "action": "\u0e40\u0e23\u0e34\u0e48\u0e21\u0e01\u0e32\u0e23\u0e2a\u0e19\u0e17\u0e19\u0e32\u0e14\u0e49\u0e27\u0e22\u0e01\u0e32\u0e23\u0e17\u0e31\u0e01\u0e17\u0e32\u0e22\u0e2d\u0e22\u0e48\u0e32\u0e07\u0e21\u0e37\u0e2d\u0e2d\u0e32\u0e0a\u0e35\u0e1e\u0e41\u0e25\u0e30\u0e41\u0e19\u0e30\u0e19\u0e33\u0e15\u0e31\u0e27",
    "explanation": "\u0e40\u0e23\u0e34\u0e48\u0e21\u0e01\u0e32\u0e23\u0e42\u0e17\u0e23\u0e42\u0e14\u0e22\u0e01\u0e32\u0e23\u0e2a\u0e23\u0e49\u0e32\u0e07\u0e04\u0e27\u0e32\u0e21\u0e19\u0e48\u0e32\u0e40\u0e0a\u0e37\u0e48\u0e2d\u0e16\u0e37\u0e2d\u0e41\u0e25\u0e30\u0e2a\u0e23\u0e49\u0e32\u0e07\u0e04\u0e27\u0e32\u0e21\u0e2a\u0e31\u0e21\u0e1e\u0e31\u0e19\u0e18\u0e4c\u0e01\u0e31\u0e1a\u0e25\u0e39\u0e01\u0e04\u0e49\u0e32",
    "signals": [
      "\u0e40\u0e1e\u0e34\u0e48\u0e21\u0e40\u0e23\u0e34\u0e48\u0e21\u0e15\u0e49\u0e19\u0e01\u0e32\u0e23\u0e42\u0e17\u0e23",
      "\u0e25\u0e39\u0e01\u0e04

In [68]:
from package.program.extraction import extract_information, extract_customer_data

extract_information(model_id, "ผมสมชายอายุ 25 ปี")

CustomerInfo(age=25, income_per_month=None, marital_status=None, number_of_children=None)

In [ ]:
await extract_customer_data(model_id, "ผมสมชายอาย 25 ปี")

In [49]:
client.close()

<coroutine object WSClient.close at 0x0000020731405F00>

In [36]:
client = WSClient()
await client.connect()

In [37]:
response1 = await client.send_and_wait("Test")
response1

'{"type": "guide", "stage_name": "greeting", "guide": {"action": "\\u0e40\\u0e23\\u0e34\\u0e48\\u0e21\\u0e01\\u0e32\\u0e23\\u0e2a\\u0e19\\u0e17\\u0e19\\u0e32\\u0e14\\u0e49\\u0e27\\u0e22\\u0e01\\u0e32\\u0e23\\u0e17\\u0e31\\u0e01\\u0e17\\u0e32\\u0e22\\u0e2d\\u0e22\\u0e48\\u0e32\\u0e07\\u0e21\\u0e37\\u0e2d\\u0e2d\\u0e32\\u0e0a\\u0e35\\u0e1e\\u0e41\\u0e25\\u0e30\\u0e41\\u0e19\\u0e30\\u0e19\\u0e33\\u0e15\\u0e31\\u0e27", "explanation": "\\u0e40\\u0e23\\u0e34\\u0e48\\u0e21\\u0e01\\u0e32\\u0e23\\u0e42\\u0e17\\u0e23\\u0e42\\u0e14\\u0e22\\u0e01\\u0e32\\u0e23\\u0e2a\\u0e23\\u0e49\\u0e32\\u0e07\\u0e04\\u0e27\\u0e32\\u0e21\\u0e19\\u0e48\\u0e32\\u0e40\\u0e0a\\u0e37\\u0e48\\u0e2d\\u0e16\\u0e37\\u0e2d\\u0e41\\u0e25\\u0e30\\u0e2a\\u0e23\\u0e49\\u0e32\\u0e07\\u0e04\\u0e27\\u0e32\\u0e21\\u0e2a\\u0e31\\u0e21\\u0e1e\\u0e31\\u0e19\\u0e18\\u0e4c\\u0e01\\u0e31\\u0e1a\\u0e25\\u0e39\\u0e01\\u0e04\\u0e49\\u0e32", "signals": ["\\u0e40\\u0e1e\\u0e34\\u0e48\\u0e21\\u0e40\\u0e23\\u0e34\\u0e48\\u0e21\\u0e15\\u0e49\\u

In [38]:
response2 = await client.send_and_wait("ผมแบงค์จากบริษัทประกันชีวิต")
response2

'{"status": "ok"}'

In [33]:
client.close()

<coroutine object WSClient.close at 0x0000020730CBC100>

In [70]:
# In dev.ipynb
import asyncio
import websockets
import json

class WSClient:
    def __init__(self, uri="ws://localhost:8000/ws"):
        self.uri = uri
        self.websocket = None
    
    async def connect(self):
        self.websocket = await websockets.connect(self.uri)
        # Skip initial greeting
        initial = await self.websocket.recv()
        print("📥 Initial greeting received (skipped)")
        return initial

    async def close(self):
        await self.websocket.close()

async def test_extraction_with_console_output():
    print("🚀 Starting WebSocket extraction test...")
    
    client = WSClient()
    await client.connect()
    
    # Send guide command
    command = {
        "type": "guide",
        "data": {
            "stage_name": "discovery", 
            "content": "ผมชื่อสมชายอายุ 25 ปี มีเงินเดือน 30000 บาท ผมสนใจทำประกันชีวิต แต่ไม่สนใจประกันสุขภาพครับ"
        }
    }
    
    print(f"📤 Sending command: {json.dumps(command, ensure_ascii=False)}")
    await client.websocket.send(json.dumps(command))
    
    print("\n📥 Waiting for responses...")
    print("=" * 60)
    
    # Listen for responses
    responses = []
    for i in range(5):  # Wait for up to 5 responses
        try:
            print(f"⏳ Waiting for response {i+1}...")
            response = await asyncio.wait_for(client.websocket.recv(), timeout=20.0)
            data = json.loads(response)
            responses.append(data)
            
            print(f"✅ Response {i+1} received:")
            print(f"   Type: {data['type']}")
            print(f"   Full response: {json.dumps(data, ensure_ascii=False, indent=2)}")
            print("-" * 40)
            
        except asyncio.TimeoutError:
            print(f"⏰ Timeout waiting for response {i+1}")
            break
        except Exception as e:
            print(f"❌ Error: {e}")
            break
    
    await client.close()
    
    print(f"\n🎯 Test completed! Total responses: {len(responses)}")
    print("=" * 60)
    
    # Summary
    for i, resp in enumerate(responses):
        print(f"{i+1}. {resp['type']}")
    
    return responses

# Run the test
print("🔥 STARTING EXTRACTION TEST")
print("Check your FastAPI server console for debug messages!")
print("=" * 60)

responses = await test_extraction_with_console_output()


🔥 STARTING EXTRACTION TEST
Check your FastAPI server console for debug messages!
🚀 Starting WebSocket extraction test...
📥 Initial greeting received (skipped)
📤 Sending command: {"type": "guide", "data": {"stage_name": "discovery", "content": "ผมชื่อสมชายอายุ 25 ปี มีเงินเดือน 30000 บาท ผมสนใจทำประกันชีวิต แต่ไม่สนใจประกันสุขภาพครับ"}}

📥 Waiting for responses...
⏳ Waiting for response 1...
✅ Response 1 received:
   Type: guide
   Full response: {
  "type": "guide",
  "stage_name": "discovery",
  "message": "Processing conversation for discovery stage",
  "status": "processing"
}
----------------------------------------
⏳ Waiting for response 2...
✅ Response 2 received:
   Type: information
   Full response: {
  "type": "information",
  "customer_information": {
    "age": 25,
    "income_per_month": 30000,
    "marital_status": null,
    "number_of_children": null
  }
}
----------------------------------------
⏳ Waiting for response 3...
✅ Response 3 received:
   Type: interest
   Ful

In [71]:
# In dev.ipynb
from package.program.memory import conversation_memory

# Test initial state
print("🔍 Initial State:")
print(conversation_memory.get_current_state())

# Test customer information updates
print("\n📊 Testing Customer Information Updates:")
result1 = conversation_memory.update_customer_information({"age": 25, "name": "สมชาย"})
print(f"First update result: {result1}")
print(f"Current info: {conversation_memory.customer_information}")

# Test same data (should not update)
result2 = conversation_memory.update_customer_information({"age": 25, "name": "สมชาย"})
print(f"Same data update result: {result2}")

# Test partial update
result3 = conversation_memory.update_customer_information({"income": 30000})
print(f"Partial update result: {result3}")
print(f"Updated info: {conversation_memory.customer_information}")

# Test customer interests
print("\n🎯 Testing Customer Interest Updates:")
result4 = conversation_memory.update_customer_interest({"life_insurance": True, "health_insurance": False})
print(f"Interest update result: {result4}")
print(f"Current interests: {conversation_memory.customer_interest}")

# Test agent checklist
print("\n📋 Testing Agent Checklist:")
result5 = conversation_memory.update_agent_checklist({"agent_introduced": True, "company_mentioned": False})
print(f"Checklist update result: {result5}")
print(f"Is complete: {conversation_memory.is_checklist_complete()}")

# Complete checklist
result6 = conversation_memory.update_agent_checklist({"company_mentioned": True, "permission_asked": True})
print(f"Complete checklist result: {result6}")
print(f"Is complete now: {conversation_memory.is_checklist_complete()}")

# Test stage transitions
print("\n🔄 Testing Stage Transitions:")
stage_result = conversation_memory.update_stage("discovery")
print(f"Stage change result: {stage_result}")
print(f"Current stage: {conversation_memory.current_stage}")
print(f"Previous stage: {conversation_memory.previous_stage}")

# Final state
print("\n🎯 Final State:")
print(conversation_memory.get_current_state())


🔍 Initial State:
{'current_stage': 'greeting', 'previous_stage': None, 'agent_checklist': {}, 'customer_information': {}, 'customer_interest': {}, 'product_count': 0}

📊 Testing Customer Information Updates:
👤 Customer info updated: age = None → 25
👤 Customer info updated: name = None → สมชาย
First update result: True
Current info: {'age': 25, 'name': 'สมชาย'}
Same data update result: False
👤 Customer info updated: income = None → 30000
Partial update result: True
Updated info: {'age': 25, 'name': 'สมชาย', 'income': 30000}

🎯 Testing Customer Interest Updates:
🎯 Customer interest updated: life_insurance = None → True
🎯 Customer interest updated: health_insurance = None → False
Interest update result: True
Current interests: {'life_insurance': True, 'health_insurance': False}

📋 Testing Agent Checklist:
📋 Checklist updated: agent_introduced = True
📋 Checklist updated: company_mentioned = False
Checklist update result: True
Is complete: False
📋 Checklist updated: company_mentioned = True

In [72]:
from package.program.extraction import suggest_objection

In [74]:
suggest_objection(model_id, "ผมว่าผมขอปรึกษาภรรยาก่อนน่าจะดีกว่าครับ")

Guide(action='ยอมรับและชี้แจงให้ลูกค้ารอ', explanation='การแสดงความต้องการให้คำปรึกษาพร้อมคนคุณโปรดแต่สามารถนำเสนอคุณค่าเพิ่มเติมได้เมื่อคุณกลับมา', signals=['ลูกค้าต้องการปรึกษาคู่ชีวิตก่อนตัดสินใจ', 'ลูกค้าแสดงความไม่แน่ใจ'], lines_to_say=['ไม่เป็นไรเลย ลูกค้าคุณสามารถรอไปกับผมได้ ฉันรออยู่ที่นี่ แต่ก่อนอยู่แล้วขอให้คุณคิดคุณค่าที่เราได้นำเสนอไปหนักนานกว่านี้'])

In [75]:
conversation_memory

ConversationMemory(current_stage='discovery', previous_stage='greeting', agent_checklist={'agent_introduced': True, 'company_mentioned': True, 'permission_asked': True}, customer_information={'age': 25, 'name': 'สมชาย', 'income': 30000}, customer_interest={'life_insurance': True, 'health_insurance': False}, product_list=[])

In [77]:
# Test Customer Extraction and Product Filtering
import pandas as pd
import json
from package.program.extraction import extract_information, extract_interest
from package.program.memory import conversation_memory
from package.program.product_filter import filter_and_send_products, product_filter_by_params

# Load products dataset
products_df = pd.read_csv("./dataset/mock_life_insurance_products.csv")
print(f"📊 Loaded {len(products_df)} products")

# Test conversation content
test_conversation = """
Agent: สวัสดีครับ ผมชื่อจอห์น จากบริษัทประกันชีวิต ABC ครับ
Customer: สวัสดีค่ะ 
Agent: วันนี้ผมโทรมาเพื่อแนะนำโปรดักส์ประกันชีวิตใหม่ของเราครับ คุณมีเวลาสักครู่ไหมครับ
Customer: ได้ค่ะ ตอนนี้ดิฉันอายุ 35 ปี แต่งงานแล้ว มีลูก 2 คน รายได้เดือนละ 50,000 บาท กำลังมองหาประกันชีวิตสำหรับครอบครัวอยู่พอดี
Agent: ดีมากครับ คุณสนใจประกันแบบไหนครับ
Customer: อยากได้ประกันชีวิต ประกันสุขภาพ และการวางแผนเกษียณค่ะ ไม่สนใจประกันอุบัติเหตุ
"""

# Test extraction functions (synchronous)
print("🧠 Testing AI Extraction...")

# Extract customer information
print("\n1️⃣ Extracting Customer Information...")
info_result = extract_information("us.amazon.nova-micro-v1:0", test_conversation)
print(f"📋 Extracted Info: {info_result}")

# Extract customer interests  
print("\n2️⃣ Extracting Customer Interests...")
interest_result = extract_interest("us.amazon.nova-micro-v1:0", test_conversation)
print(f"💡 Extracted Interests: {interest_result}")

# Update conversation memory
print("\n🧠 Updating Conversation Memory...")
if info_result:
    conversation_memory.update_customer_information(info_result.dict())
    print(f"✅ Updated customer info: {conversation_memory.customer_information}")

if interest_result:
    conversation_memory.update_customer_interest(interest_result.dict())
    print(f"✅ Updated customer interests: {conversation_memory.customer_interest}")

# Test product filtering
print("\n🛍️ Testing Product Filtering...")

# Get current filters from memory
filters = {}
if 'age' in conversation_memory.customer_information:
    filters['age'] = conversation_memory.customer_information['age']
if 'income_per_month' in conversation_memory.customer_information:
    filters['income_per_month'] = conversation_memory.customer_information['income_per_month']

print(f"🔍 Filtering with: {filters}")

# Filter products
if filters:
    filtered_products = product_filter_by_params(products_df, 0.2, **filters)
    print(f"📦 Found {len(filtered_products)} matching products")
    
    # Show filtered products
    target_columns = [
        "product_id", "product_name", "objective", 
        "premium_min_month_thb", "premium_max_month_thb", 
        "age_min", "age_max", "notes"
    ]
    
    products_list = filtered_products[target_columns].to_dict(orient="records")
    
    print("\n🎯 Filtered Products:")
    for i, product in enumerate(products_list[:3], 1):  # Show first 3
        print(f"\n{i}. {product['product_name']}")
        print(f"   💰 Premium: {product['premium_min_month_thb']:,}-{product['premium_max_month_thb']:,} THB/month")
        print(f"   👥 Age: {product['age_min']}-{product['age_max']} years")
        print(f"   📝 {product['notes']}")
    
    # Test memory update
    is_updated = conversation_memory.update_product_list(products_list)
    print(f"\n✅ Product list updated in memory: {is_updated}")
    
    # Show final WebSocket message format
    websocket_message = {
        "type": "products",
        "products": products_list,
        "count": len(products_list)
    }
    
    print(f"\n📡 WebSocket Message Preview:")
    print(json.dumps(websocket_message, indent=2, ensure_ascii=False)[:500] + "...")
    
else:
    print("❌ No customer data available for filtering")

# Show memory state
print(f"\n🧠 Final Memory State:")
print(f"Customer Info: {conversation_memory.customer_information}")
print(f"Customer Interests: {conversation_memory.customer_interest}")
print(f"Product Count: {len(conversation_memory.product_list)}")
                        

📊 Loaded 28 products
🧠 Testing AI Extraction...

1️⃣ Extracting Customer Information...
📋 Extracted Info: age=35 income_per_month=50000 marital_status='Married' number_of_children=2

2️⃣ Extracting Customer Interests...
💡 Extracted Interests: life_insurance=True health_insurance=True critical_illness=None accident_insurance=False retirement_planning=True tax_benefits=None

🧠 Updating Conversation Memory...
👤 Customer info updated: age = 25 → 35
👤 Customer info updated: income_per_month = None → 50000
👤 Customer info updated: marital_status = None → Married
👤 Customer info updated: number_of_children = None → 2
✅ Updated customer info: {'age': 35, 'name': 'สมชาย', 'income': 30000, 'income_per_month': 50000, 'marital_status': 'Married', 'number_of_children': 2}
🎯 Customer interest updated: health_insurance = False → True
🎯 Customer interest updated: accident_insurance = None → False
🎯 Customer interest updated: retirement_planning = None → True
✅ Updated customer interests: {'life_insura

C:\Users\Admin\AppData\Local\Temp\ipykernel_30548\3749042055.py:38: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  conversation_memory.update_customer_information(info_result.dict())
C:\Users\Admin\AppData\Local\Temp\ipykernel_30548\3749042055.py:42: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  conversation_memory.update_customer_interest(interest_result.dict())
